# Regression multiple (OLS)

Application d'une régression linéaire multiple sur un jeu de données e-commerce, avec une attention particulière portée aux tests qui accompagnent ce type de modèle : significativité globale (Fisher), significativité des coefficients (Student), et vérification de la multicolinéarité (VIF).

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set(style="whitegrid")
    
except:
    sns = None
# chargement du jeu de données 
df = pd.read_csv("ecommerce_dataset.csv")
df.head()

,client_id,sexe,âge,revenu_mensuel,segment_age,revenu_cat,produit,canal_achat,montant_panier,panier_frequent,note_satisfaction,retour_produit,date_achat
0,1,Homme,50,3893.70,Mature,Moyen,Sport,Mobile,83.27,Non,2,Non,2023-05-28
1,2,Femme,39,2885.15,Adulte,Moyen,Électronique,Mobile,47.18,Oui,3,Non,2023-01-13
2,3,Homme,44,2369.53,Mature,Moyen,Sport,Mobile,72.28,Oui,4,Non,2023-03-13
3,4,Homme,38,3197.92,Adulte,Moyen,Électronique,Mobile,54.59,Non,3,Non,2023-05-07
4,5,Homme,20,3774.66,Jeune,Moyen,Sport,Mobile,75.30,Non,4,Non,2023-07-04


Question :  
> Le montant du panier dépend-il du revenu, du sexe, de la tranche d'âge, du canal d'achat, de la categeorie, de la saison ou encore de la frequence panier ?

# Choix des variables

- `revenu_cat` est écartée : elle contient la même information que 
  `revenu_mensuel`, sous forme de tranches. Garder les deux créerait une colinéarité quasi parfaite.
- `note` est écartée : c'est une note de satisfaction laissée après 
  l'achat. Elle est la conséquence du montant dépensé, pas une cause. L'inclure inverserait le sens de la relation cherchée.
- `date` est transformée en `saison` (4 modalités) plutôt que gardée brute. Une date brute aurait trop de valeurs uniques pour être exploitable en dummies.

In [11]:
# conversion de la colonne date en type datetime
df["date_achat"] = pd.to_datetime(df["date_achat"])

# extraction du mois
df["mois"] = df["date_achat"].dt.month

# fonction qui associe un mois à une saison
def mois_vers_saison(mois):
    if mois in [12, 1, 2]:
        return "Hiver"
    elif mois in [3, 4, 5]:
        return "Printemps"
    elif mois in [6, 7, 8]:
        return "Été"
    else:
        return "Automne"

# application de la fonction pour créer la colonne saison
df["saison"] = df["mois"].apply(mois_vers_saison)

# vérif de la répartition des saisons
df["saison"].value_counts()

saison
Été          3824
Printemps    3764
Automne      3761
Hiver        3651
Name: count, dtype: int64

In [12]:
# verif  du nombre de modalités pour la transformation en dummies
df["produit"].nunique() 

 # détail
df["produit"].value_counts()

produit
Électronique    3684
Maison          3125
Mode            2966
Beauté          2951
Sport           2274
Name: count, dtype: int64

### Encodage des variables catégorielles

Chaque variable catégorielle est transformée en dummies avec `drop_first=True`. Une modalité est retirée pour chaque variable. Ça évite une colinéarité parfaite entre les dummies et la constante.

La modalité retirée devient la référence. Elle n'a pas de coefficient propre. Elle est absorbée dans la constante (β₀). Les coefficients des autres modalités se lisent comme un écart par rapport à cette références.

Ex : Pour `segment_age`, la modalité retirée est `Adulte`. Le coefficient de `Senior`, par exemple, indique l'écart moyen de montant_panier entre les Seniors et les Adultes. Pas entre les Seniors et les autres segments pris ensemble. 

C'est différent de l'ANOVA, qui compare toutes les moyennes entre elles sans notion de référence.

In [13]:
#transfomations des colonnes en dummies 
dummies = pd.get_dummies(df[["sexe", "segment_age", "canal_achat", "panier_frequent", "produit", "saison"]], drop_first=True)
dummies.head()

,sexe_Homme,segment_age_Jeune,segment_age_Mature,segment_age_Senior,canal_achat_Desktop,canal_achat_Email,canal_achat_Mobile,panier_frequent_Oui,produit_Maison,produit_Mode,produit_Sport,produit_Électronique,saison_Hiver,saison_Printemps,saison_Été
0,True,False,True,False,False,False,True,False,False,False,True,False,False,True,False
1,False,False,False,False,False,False,True,True,False,False,False,True,True,False,False
2,True,False,True,False,False,False,True,True,False,False,True,False,False,True,False
3,True,False,False,False,False,False,True,False,False,False,False,True,False,True,False
4,True,True,False,False,False,False,True,False,False,False,True,False,False,False,True


In [14]:
# ajout de la variable quantitative pour etablir la matrice
X = pd.concat([df[["revenu_mensuel"]], dummies], axis=1)
X.head()

,revenu_mensuel,sexe_Homme,segment_age_Jeune,segment_age_Mature,segment_age_Senior,canal_achat_Desktop,canal_achat_Email,canal_achat_Mobile,panier_frequent_Oui,produit_Maison,produit_Mode,produit_Sport,produit_Électronique,saison_Hiver,saison_Printemps,saison_Été
0,3893.70,True,False,True,False,False,False,True,False,False,False,True,False,False,True,False
1,2885.15,False,False,False,False,False,False,True,True,False,False,False,True,True,False,False
2,2369.53,True,False,True,False,False,False,True,True,False,False,True,False,False,True,False
3,3197.92,True,False,False,False,False,False,True,False,False,False,False,True,False,True,False
4,3774.66,True,True,False,False,False,False,True,False,False,False,True,False,False,False,True


# Regression

In [15]:
import statsmodels.api as sm

# ajout de la constante (β0) sm.OLS ne l'ajoute pas automatiquement
X_const = sm.add_constant(X)

y = df["montant_panier"]
X_const = X_const.astype(float) #conversion des variables

modele = sm.OLS(y, X_const.astype(float)).fit()
print(modele.summary())

                            OLS Regression Results                            
Dep. Variable:         montant_panier   R-squared:                       0.061
Model:                            OLS   Adj. R-squared:                  0.060
Method:                 Least Squares   F-statistic:                     60.56
Date:                Tue, 29 Sep 2026   Prob (F-statistic):          1.11e-189
Time:                        20:33:29   Log-Likelihood:                -83718.
No. Observations:               15000   AIC:                         1.675e+05
Df Residuals:                   14983   BIC:                         1.676e+05
Df Model:                          16                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                   50.9865 

## Interprétation  

Le modèle est globalement significatif (F = 60,56 ; p = 1,11e-189) et explique une part réelle de la variance des dépenses. Mais le R² reste à 6,1 %. Les variables retenues expliquent très peu la dépense totale d'un client. Il manque sans doute des facteurs importants dans le modèle.

Les clients achetant dans la catégorie : 
- `Électronique` dépensent en moyenne 34,71 € 
- `Maison` dépensent en moyenne 23,05 €
- `Mode`dépensent en moyenne 12,14 €
  
de plus que les clients achetant dans la catégorie 'Beauté' (modalité de réf).

Les trois écarts sont significatifs (p<0,001). La catégorie `Sport` ne se distingue pas: IC 95 % de −3,80 à +5,33 €.

Le revenu mensuel a aussi un effet significatif mais faible. Chaque euro de revenu en plus ajoute 0,0097 € seulement de dépense.

Les autres variables `sexe`, `âge`, `canal d'achat`, `fréquence du panier`, `saison` n'ont pas d'effet significatif. Leurs p-values dépassent largement 0,05, et leurs intervalles de confiance contiennent tous zéro.

### Langage metier 

Le meilleur prédicteur du montant est la catégorie de produit.

Aucune différence détectable entre Sport et Beauté.
Le revenu joue un rôle, mais faible. 1000 € de revenu mensuel en plus ne rapportent que 9,70 € de dépense en plus. L'effet est faible.

Le sexe, l'âge, le canal d'achat, la fréquence d'achat et la saison n'ont pas d'effet détectable dans ce modèle. Segmenter une campagne sur ces critères ne changerait rien au panier moyen.

Pour l'entreprise,la catégorie de **produit** reste le meilleur prédicteur du montant du panier. Les clients ***Électronique et Maison*** dépensent le **plus**. Peut justifier de leur donner la priorité dans les actions marketing (à confirmer avec les marges). Encore une fois, le modèle ne capte qu'une petite partie de ce qui explique la dépense. D'autres données seraient nécessaires pour un vrai outil de prédiction (historique d'achat, fidélité, panier moyen passé).



# VIF

In [16]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd


# le VIF se calcule sur la même matrice que celle utilisée pour la régression
# (X_const contient déjà la constante et les variables catégorielles avec la modalité de référence retirée)
vif_data = pd.DataFrame()
# on récupère le nom de chaque variable
vif_data["variable"] = X_const.columns

vif_data["VIF"] = [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]

print(vif_data)

                variable        VIF
0                  const  30.888919
1         revenu_mensuel   1.053473
2             sexe_Homme   2.041835
3      segment_age_Jeune   1.084505
4     segment_age_Mature   1.091857
5     segment_age_Senior   1.011897
6    canal_achat_Desktop   2.911022
7      canal_achat_Email   2.508824
8     canal_achat_Mobile   3.153782
9    panier_frequent_Oui   1.061489
10        produit_Maison   1.970709
11          produit_Mode   1.677082
12         produit_Sport   2.531964
13  produit_Électronique   2.663724
14          saison_Hiver   1.492374
15      saison_Printemps   1.499596
16            saison_Été   1.503440


## Interprétation

Aucun VIF ne dépasse 3,2,pas de multicolinéarité problématique.

Le canal d'achat a les VIF les plus élevés : Mobile 3,15, Desktop 2,91, Email 2,51. Les dummies d'une même variable s'excluent entre elles, donc elles sont corrélées mécaniquement. Les catégories de produit suivent la même logique (de 1,68 à 2,66).

Le sexe a un VIF de 2,04. Environ la moitié de sa variance s'explique par les autres variables, sans doute par les catégories de produit. Un Khi² sexe × produit permettrait de le vérifier.

Le revenu (1,05), le segment d'âge (1,01 à 1,09), la fréquence de panier (1,06) et la saison (1,49 à 1,50) sont presque indépendants du reste.

Le VIF de la constante (30,9) ne s'interprète pas. 

### Implication métier
Les coefficients sont estimés sans distorsion liée à la colinéarité. 
Le faible R² (6,1 %) tient au manque de variables capables d'expliquer la dépense. : historique d'achat, fidélité, panier moyen passé.

## Khi-deux : sexe x produit

In [19]:
# Tableau croisé sexe × produit
table_sp = pd.crosstab(df["sexe"], df["produit"])
print(table_sp)

# Proportions par ligne autrement dit quelle part de chaque sexe achète chaque produit
table_sp_prop = table_sp.div(table_sp.sum(axis=1), axis=0) * 100
print(table_sp_prop.round(1))

# Khi² d'indépendance
from scipy.stats import chi2_contingency
chi2, p, dof, expected = chi2_contingency(table_sp)
print(f"Khi² = {chi2:.3f}, p = {p:.3e}, ddl = {dof}")

# V de Cramér
n = table_sp.to_numpy().sum()
phi2 = chi2 / n
r, k = table_sp.shape
phi2_corr = max(0, phi2 - (k-1)*(r-1)/(n-1))
r_corr = r - (r-1)**2/(n-1)
k_corr = k - (k-1)**2/(n-1)
v = np.sqrt(phi2_corr / min(k_corr-1, r_corr-1))
print(f"V de Cramér : {v:.3f}")

produit  Beauté  Maison  Mode  Sport  Électronique
sexe                                              
Femme      2951    1540  2281      0           707
Homme         0    1585   685   2274          2977
produit  Beauté  Maison  Mode  Sport  Électronique
sexe                                              
Femme      39.5    20.6  30.5    0.0           9.5
Homme       0.0    21.1   9.1   30.2          39.6
Khi² = 7483.118, p = 0.000e+00, ddl = 4
V de Cramér : 0.706


Le sexe est fortement lié au produit acheté (Khi² = 7483, p < 0,001, V de Cramér = 0,71). Aucun homme n'achète de Beauté, aucune femme de Sport, et les hommes achètent surtout de l'Électronique, la catégorie la plus chère. À produit égal (et à autres variables égales), le sexe n'a pas d'effet détectable sur le panier, le coefficient est de −1,26 € et n'est pas significatif (p = 0,40). 
L'écart brut de 10,71 € passe donc par le choix du produit. Le sexe est associé ce qui est acheté, et le produit détermine le montant.

Implication métier : Le sexe n'explique pas directement le montant, mais il prédit le type de produit acheté, et donc indirectement le panier.

# Lmites

Le summary du modèle donne deux informations. Le skewness est de 2,09 et le kurtosis de 10,64. Une distribution normale a un skew proche de 0 et un kurtosis proche de 3. Les résidus s'éloignent fortement de la normalité.

Le condition number est de 3,01e+04. C'est élevé. Mais le VIF ne montre aucun problème de multicolinéarité, toutes les valeurs sont sous 5. L'écart vient probablement d'un effet d'échelle. Le revenu mensuel prend des valeurs de plusieurs milliers d'euros. Les autres variables sont des dummies (0 ou 1). Ce n'est pas un vrai signal de colinéarité, mais un point à noter.

Le modèle reste limité. Le R² à 6 % le confirme. La dépense d'un client dépend surtout de facteurs absents du modèle.